In [4]:
import pandas as pd
import numpy as np
import random

class MyLineReg():

    def __init__(self, n_iter = 10, learning_rate = None, weights = None, metric = None, reg = None, l1_coef = 0, l2_coef = 0, sgd_sample = None, random_state = 42):
        self.n_iter = n_iter
        self.learning_rate = learning_rate
        self.weights = weights
        self.metric = metric
        self.best_metric = None
        self.reg = reg
        self.l1_coef = l1_coef
        self.l2_coef = l2_coef
        self.sgd_sample = sgd_sample
        self.random_state = random_state

    def fit(self, X, y, verbose = False):
        X_with_bias = X.copy()
        X_with_bias.insert(0, 'x0', 1)
        X_matrix = X_with_bias.values
        y_vector = y.values
        n_samples, counts_of_features = X_matrix.shape
        random.seed(self.random_state)

        if self.weights is None:
            self.weights = np.ones(counts_of_features)

        for i in range(1, self.n_iter + 1):
            y_cup = X_matrix.dot(self.weights)

            if self.sgd_sample:
                if self.sgd_sample <= 1.0:
                    batch_size = int(n_samples * self.sgd_sample)
                else:
                    batch_size = self.sgd_sample
                sample_rows_idx = random.sample(range(X.shape[0]), batch_size)
                X_sgd = X_matrix[sample_rows_idx]
                y_sgd = y_vector[sample_rows_idx]
                y_sgd_cup = X_sgd.dot(self.weights)


            if self.reg == 'l1':
                self.l2_coef = 0
            elif self.reg == 'l2':
                self.l1_coef = 0
            elif self.reg is None:
                self.l1_coef = self.l2_coef = 0

            loss = np.mean((y_vector - y_cup) ** 2) \
                    + self.l1_coef * np.sum(self.weights) \
                    + self.l2_coef * np.sum(self.weights ** 2)
            if self.sgd_sample != None:
                grad_loss = 2 * X_sgd.T.dot(y_sgd_cup - y_sgd) / batch_size\
                    + self.l1_coef * (np.sign(self.weights))\
                    + self.l2_coef * self.weights * 2
            else:
                grad_loss = 2 * X_matrix.T.dot(y_cup - y_vector) / n_samples\
                    + self.l1_coef * (np.sign(self.weights))\
                    + self.l2_coef * self.weights * 2

            if isinstance(self.learning_rate, (float, int)):
                self.weights = self.weights - self.learning_rate * grad_loss
            else:
                self.weights = self.weights - self.learning_rate(i) * grad_loss

            if self.metric != None:
                metric_func = getattr(self, self.metric)
                self.best_metric = metric_func(X_matrix, y_vector)
                if verbose:
                    if i % verbose == 0:
                        print(f"step {i}| loss = {loss} | metric = {self.best_metric}")

        return

    def get_coef(self):
        return self.weights[1:].copy()

    def predict(self, X_matrix):
        X_with_bias = X_matrix.copy()
        X_with_bias.insert(0, 'x0', 1)
        X_vals = X_with_bias.values
        return X_vals.dot(self.weights)

    def mse(self, X_matrix, y_vector):
        y_cup = X_matrix.dot(self.weights)
        return np.mean((y_vector - y_cup) ** 2)

    def mae(self, X_matrix, y_vector):
        y_cup = X_matrix.dot(self.weights)
        return np.mean(np.abs(y_vector - y_cup))

    def rmse(self, X_matrix, y_vector):
        y_cup = X_matrix.dot(self.weights)
        return np.sqrt(np.mean((y_vector - y_cup) ** 2))

    def r2(self, X_matrix, y_vector):
        y_cup = X_matrix.dot(self.weights)
        y_mean = np.mean(y_vector)
        return (1 - (np.sum((y_vector - y_cup) ** 2) /
                    np.sum((y_vector - y_mean) ** 2)))

    def mape(self, X_matrix, y_vector):
        y_cup = X_matrix.dot(self.weights)
        return np.mean(100 * np.abs((y_vector - y_cup)) /
                       np.abs(y_vector))

    def get_best_score(self):
        return self.best_metric


    def __str__(self):
        return f"MyLineReg class: n_iter={self.n_iter}, learning_rate={self.learning_rate}"
